# MRI-Xplain: An Agentic AI Framework for Faithful Explanations in Brain Tumour MRI Classification (Demo)

Imane Belbachir |
MSc Computer Science with Artificial Intelligence |
MSc Dissertation |
Abertay University |
July 2026

# Step 1: Environment Setup and Classification Model Loading

Load the required libraries, configure the runtime environment, define the project settings, and load the trained EfficientNet-B0 classification model for subsequent AgentXAI experiments.

In [1]:
from google.colab import drive
import os, warnings, json, pickle
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
warnings.filterwarnings('ignore')

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/AgentXAI_MSc'

device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES   = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IOU_THRESHOLD = 0.25   # empirical threshold — used everywhere, never changes
MAX_ITER      = 5      # maximum XAI retry attempts per image

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

#  Model
def build_efficientnet_b0(num_classes=4, dropout=0.3):
    m = models.efficientnet_b0(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, num_classes))
    return m.to(device)

eff_model = build_efficientnet_b0()
eff_model.load_state_dict(
    torch.load(f'{DRIVE_DIR}/EfficientB0_NoSmoothing_weights.pth', map_location=device)
)
eff_model.eval()
print(f"EfficientNetB0 loaded on {device}")

Mounted at /content/drive
EfficientNetB0 loaded on cuda


# Step 2: Dataset Preparation and Sample Selection

Load the BRISC test dataset, prepare the MRI images and segmentation masks, and select one representative sample from each class for AgentXAI evaluation.

In [2]:
import glob
from torch.utils.data import Dataset

DRIVE_DATA_ZIP = '/content/drive/MyDrive/Medical_Data/brisc2025.zip'
LOCAL_EXTRACT  = '/content/brisc2025_dataset'

if not os.path.exists(LOCAL_EXTRACT):
    os.makedirs(LOCAL_EXTRACT, exist_ok=True)
    os.system(f'unzip -q "{DRIVE_DATA_ZIP}" -d "{LOCAL_EXTRACT}"')

found = glob.glob("/content/**/classification_task", recursive=True)
BRISC_BASE_DIR = os.path.dirname(found[0])

class BRISCDataset(Dataset):
    def __init__(self, base_dir, split='test', transform=None):
        self.transform    = transform
        self.class_dir    = os.path.join(base_dir, 'classification_task', split)
        self.seg_mask_dir = os.path.join(base_dir, 'segmentation_task', split, 'masks')
        self.classes      = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples      = []
        self._build_index()

    def _build_index(self):
        for cls in self.classes:
            folder = os.path.join(self.class_dir, cls)
            if not os.path.exists(folder):
                continue
            for img_name in os.listdir(folder):
                if not img_name.lower().endswith(('.jpg','.jpeg','.png')):
                    continue
                img_path  = os.path.join(folder, img_name)
                mask_path = os.path.join(
                    self.seg_mask_dir, os.path.splitext(img_name)[0] + '.png'
                )
                self.samples.append({
                    'image_path': img_path,
                    'label':      self.class_to_idx[cls],
                    'mask_path':  mask_path if os.path.exists(mask_path) else None,
                    'class_name': cls,
                })

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['image_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(s['label']), torch.zeros(1, 224, 224)

val_dataset = BRISCDataset(BRISC_BASE_DIR, 'test', val_transform)

# One demo sample per class
test_samples = {}
for s in val_dataset.samples:
    if s['label'] not in test_samples:
        test_samples[s['label']] = s
    if len(test_samples) == 4:
        break

print(f"Dataset: {len(val_dataset)} test images")
for i in range(4):
    print(f"  [{i}] {CLASS_NAMES[i]:<12} "
          f"mask={'yes' if test_samples[i]['mask_path'] else 'no'}")

Dataset: 1000 test images
  [0] Glioma       mask=yes
  [1] Meningioma   mask=yes
  [2] No Tumor     mask=no
  [3] Pituitary    mask=yes


# Step 3: Initialise XAI and YOLO Components

Load the XAI methods and YOLO detector, and define the utility functions used for explanation generation and spatial validation.

In [3]:
!pip install grad-cam ultralytics -q

import base64, io, cv2
import numpy as np
from PIL import Image
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenGradCAM, LayerCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from ultralytics import YOLO as YOLOInfer

#  XAI configs
XAI_CONFIGS = [
    {'method':'GradCAM',         'layer_name':'features[-1]', 'layer':eff_model.features[-1]},
    {'method':'GradCAM',         'layer_name':'features[-3]', 'layer':eff_model.features[-3]},
    {'method':'GradCAMPlusPlus', 'layer_name':'features[-3]', 'layer':eff_model.features[-3]},
    {'method':'GradCAMPlusPlus', 'layer_name':'features[-5]', 'layer':eff_model.features[-5]},
    {'method':'EigenGradCAM',    'layer_name':'features[-3]', 'layer':eff_model.features[-3]},
    {'method':'LayerCAM',        'layer_name':'features[-3]', 'layer':eff_model.features[-3]},
]
CAM_MAP = {'GradCAM':GradCAM,'GradCAMPlusPlus':GradCAMPlusPlus,
           'EigenGradCAM':EigenGradCAM,'LayerCAM':LayerCAM}

def tensor_to_rgb(tensor):
    img = tensor.cpu().numpy().transpose(1,2,0)
    return np.clip(np.array(IMAGENET_STD)*img + np.array(IMAGENET_MEAN),0,1).astype(np.float32)

def threshold_heatmap(h, pct=75, eps=1e-8):
    cut = np.percentile(h, pct)
    t   = np.where(h >= cut, h, 0.0)
    return (t - t.min()) / (t.max() - t.min() + eps)

def compute_gt_iou(heatmap, mask_path, thresh=0.5, eps=1e-8):
    if mask_path is None: return None
    mask = (np.array(Image.open(mask_path).convert('L').resize((224,224))) > 127).astype(float)
    hb   = (heatmap > thresh).astype(float)
    return float((hb*mask).sum() / (np.clip(hb+mask,0,1).sum() + eps))

def compute_yolo_iou(heatmap, yolo_mask, thresh=0.5, eps=1e-8):
    if yolo_mask is None: return None
    hb = (heatmap > thresh).astype(float)
    return float((hb*yolo_mask).sum() / (np.clip(hb+yolo_mask,0,1).sum() + eps))

def overlay_to_b64(overlay_u8):
    buf = io.BytesIO()
    Image.fromarray(overlay_u8).save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

def run_xai(cfg_idx, img_tensor, label_idx):
    cfg    = XAI_CONFIGS[cfg_idx]
    CLS    = CAM_MAP[cfg['method']]
    with CLS(model=eff_model, target_layers=[cfg['layer']]) as cam:
        h = cam(input_tensor=img_tensor,
                targets=[ClassifierOutputTarget(label_idx)])[0]
    thresh     = threshold_heatmap(h)
    img_rgb    = tensor_to_rgb(img_tensor.squeeze(0))
    overlay    = show_cam_on_image(img_rgb, thresh, use_rgb=True)
    overlay_u8 = (overlay*255).astype(np.uint8) if overlay.max()<=1.0 else overlay
    return thresh, overlay_u8, overlay_to_b64(overlay_u8), cfg

#  YOLO
yolo_detector = YOLOInfer(f'{DRIVE_DIR}/yolo_tumor_best.pt')

def run_yolo_once(image_path, img_size=224, conf=0.15):
    img_r = np.array(Image.open(image_path).convert('RGB').resize((img_size,img_size)))
    cv2.imwrite('/content/temp_yolo.jpg', cv2.cvtColor(img_r, cv2.COLOR_RGB2BGR))
    res = yolo_detector('/content/temp_yolo.jpg', imgsz=img_size, conf=conf, verbose=False)
    if not res or len(res[0].boxes)==0: return None, None
    boxes = res[0].boxes
    best  = boxes.conf.argmax().item()
    bx    = boxes.xyxy[best].cpu().numpy().astype(int)
    yc    = float(boxes.conf[best].item())
    mask  = np.zeros((img_size,img_size), dtype=np.float32)
    mask[max(0,bx[1]):min(img_size,bx[3]), max(0,bx[0]):min(img_size,bx[2])] = 1.0
    return mask, (int(bx[0]),int(bx[1]),int(bx[2]),int(bx[3]),yc)

print(f"XAI utilities ready — {len(XAI_CONFIGS)} configurations")
print(f"YOLO loaded: {yolo_detector.names}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 108.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
XAI utilities ready — 6 configurations
YOLO loaded: {0: 'tumor'}


# Step 4: Local Vision-Language Model (LLaVA-NeXT) Integration

This step loads **LLaVA-NeXT (7B)** as a local vision-language model using 4-bit quantisation for efficient inference. The model receives MRI images together with XAI explanations and generates a visual assessment that is later parsed into structured feedback for explanation validation.

In [4]:
!pip install -q --upgrade transformers accelerate bitsandbytes
import torch, io, base64, json
from PIL import Image
from transformers import (AutoProcessor,
                          LlavaNextForConditionalGeneration,
                          BitsAndBytesConfig)

model_id = "llava-hf/llava-v1.6-mistral-7b-hf"

# FIX: We pass image_token=None to override the config file's unexpected key
processor = AutoProcessor.from_pretrained(model_id, image_token=None)

llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="auto"
)

llava_model.eval()
print("LLaVA-NeXT loaded successfully!")

def query_local_llm(image_b64: str, prompt: str) -> str:
    img   = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
    conv  = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    text  = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=img, text=text, return_tensors="pt").to(llava_model.device)
    with torch.no_grad():
        out = llava_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    decoded = processor.decode(out[0], skip_special_tokens=True)
    return decoded.split("[/INST]")[-1].strip() if "[/INST]" in decoded else decoded

def parse_llm_json(raw: str) -> dict:
    cleaned = raw.replace("```json","").replace("```","").strip()
    s, e = cleaned.find("{"), cleaned.rfind("}")
    if s != -1 and e != -1:
        try: return json.loads(cleaned[s:e+1])
        except: pass
    return {"accepted":False,"reasoning":"parse failed","clinical_coherence":"low"}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 54.4 MB/s eta 0:00:00


processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

LLaVA-NeXT loaded successfully!


# Step 5: Agentic XAI Framework Implementation (MRI-Xplain)

This step implements the proposed three-agent MRI-Xplain workflow using LangGraph. The framework consists of an **XAI Selector Agent** that generates and evaluates candidate explanations, an **LLM Vision Judge Agent** that assesses clinical plausibility with YOLO-based spatial validation, and a **Clinical Reporting Agent** that produces a structured qualitative summary. The multi-agent pipeline enables iterative explanation refinement until a reliable explanation is accepted.

In [5]:
"""
The three agents exactly as described in the proposal:

Agent 1 - XAI Selector:
  Classifies the image, picks the next untried XAI config,
  generates a heatmap, computes GT-IoU.

Agent 2 - LLM Vision Judge:
  Asks LLaVA whether the highlighted region is clinically plausible.
  If LLM rejects BUT YOLO-IoU > threshold → override and accept (R8).
  If LLM rejects AND IoU low → retry with next config (up to MAX_ITER).

Agent 3 - Clinical Reporting Agent:
  Receives the accepted explanation and writes a structured summary.
  Not formally evaluated — qualitative demo only (as stated in proposal).
"""

try:
    from langgraph.graph import StateGraph, END
except ImportError:
    import subprocess; subprocess.run(["pip","install","langgraph","-q"])
    from langgraph.graph import StateGraph, END

from typing import TypedDict, Optional, List

class AgentXAIState(TypedDict):
    # Input
    image_path:              str
    true_label:              int
    mask_path:               Optional[str]
    # Classifier output
    predicted_class:         Optional[str]
    confidence:              Optional[float]
    # XAI output
    xai_method:              Optional[str]
    xai_layer:               Optional[str]
    heatmap:                 Optional[np.ndarray]
    overlay_b64:             Optional[str]
    iou_score:               Optional[float]      # vs GT mask
    # Agent control
    explanation_accepted:    Optional[bool]
    judge_reasoning:         Optional[str]
    iteration:               int
    tried_methods:           List[str]
    # YOLO spatial validator
    yolo_bbox:               Optional[tuple]
    yolo_iou:                Optional[float]      # GradCAM vs YOLO bbox
    # Provenance — for pathway evaluation
    llm_verdict_raw:         Optional[bool]       # LLM decision before override
    yolo_override_triggered: Optional[bool]
    # Agent 3 output
    abnormality_detected:    Optional[bool]
    clinical_suggestion:     Optional[str]
    final_confidence:        Optional[float]

def make_initial_state(sample: dict) -> AgentXAIState:
    return AgentXAIState(
        image_path=sample['image_path'], true_label=sample['label'],
        mask_path=sample['mask_path'],
        predicted_class=None, confidence=None,
        xai_method=None, xai_layer=None,
        heatmap=None, overlay_b64=None, iou_score=None,
        explanation_accepted=None, judge_reasoning=None,
        iteration=0, tried_methods=[],
        yolo_bbox=None, yolo_iou=None,
        llm_verdict_raw=None, yolo_override_triggered=None,
        abnormality_detected=None, clinical_suggestion=None,
        final_confidence=None,
    )

#  Agent 1: XAI Selector
def xai_selector_agent(state):
    print(f"\n  [Agent 1 — XAI Selector] iteration {state['iteration']+1}")
    img_pil    = Image.open(state['image_path']).convert('RGB')
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)
    eff_model.eval()
    with torch.no_grad():
        probs    = torch.softmax(eff_model(img_tensor), dim=1)[0]
        pred_idx = probs.argmax().item()
        conf     = probs[pred_idx].item()
    pred_class = CLASS_NAMES[pred_idx]

    tried = state.get('tried_methods', [])
    cfg_idx = next(
        (i for i,c in enumerate(XAI_CONFIGS)
         if f"{c['method']}_{c['layer_name']}" not in tried),
        len(XAI_CONFIGS)-1
    )
    thresh, _, b64, cfg = run_xai(cfg_idx, img_tensor, pred_idx)
    gt_iou = compute_gt_iou(thresh, state['mask_path'])
    key    = f"{cfg['method']}_{cfg['layer_name']}"

    # YOLO — only on first iteration; reuse cached bbox thereafter
    if state.get('yolo_bbox') is None and pred_class != 'No Tumor':
        yolo_mask, yolo_box = run_yolo_once(state['image_path'])
    elif state.get('yolo_bbox') is not None:
        x1,y1,x2,y2,_ = state['yolo_bbox']
        yolo_mask = np.zeros((224,224), dtype=np.float32)
        yolo_mask[y1:y2, x1:x2] = 1.0
        yolo_box = state['yolo_bbox']
    else:
        yolo_mask, yolo_box = None, None

    yolo_iou = compute_yolo_iou(thresh, yolo_mask)

    iou_disp = f"{gt_iou:.3f}" if gt_iou is not None else "N/A"
    yio_disp = f"{yolo_iou:.3f}" if yolo_iou is not None else "N/A"
    print(f"     Pred: {pred_class} ({conf*100:.1f}%) | "
          f"Method: {cfg['method']} @ {cfg['layer_name']} | "
          f"GT-IoU: {iou_disp} | YOLO-IoU: {yio_disp}")

    return {**state,
            'predicted_class': pred_class, 'confidence': conf,
            'xai_method': cfg['method'], 'xai_layer': cfg['layer_name'],
            'heatmap': thresh, 'overlay_b64': b64,
            'iou_score': gt_iou, 'yolo_iou': yolo_iou,
            'yolo_bbox': yolo_box,
            'tried_methods': tried + [key],
            'iteration': state['iteration']}

#  Agent 2: LLM Vision Judge
def llm_vision_judge_agent(state):
    """
    Role: evaluate whether the XAI overlay is clinically plausible.
    The LLM provides qualitative reasoning.
    The IoU override (R8 in risk register) ensures spatial evidence
    overrides LLM rejection when heatmap-tumour overlap is strong.
    """
    print(f"  [Agent 2 — LLM Vision Judge]")
    yolo_iou = state.get('yolo_iou')

    prompt = f"""You are a clinical radiology AI assistant.
A brain MRI has been classified as: {state['predicted_class']}
The GradCAM overlay shows where the model focused.
YOLO independent detector spatial IoU: {f"{yolo_iou:.3f}" if yolo_iou else "N/A"}

Is the highlighted region clinically plausible for {state['predicted_class']}?

Respond ONLY in JSON:
{{
  "accepted": true or false,
  "reasoning": "one clinical sentence describing where the highlight is and why",
  "clinical_coherence": "high/medium/low",
  "suggestion": "if rejected, where should the highlight be"
}}

Rules:
- Glioma:     highlight in brain parenchyma
- Meningioma: highlight near brain surface or meninges
- Pituitary:  highlight at skull base centre
- No Tumor:   highlight diffuse, no focal mass"""

    raw       = query_local_llm(state['overlay_b64'], prompt)
    verdict   = parse_llm_json(raw)
    llm_raw   = bool(verdict.get('accepted', False))
    reasoning = verdict.get('reasoning', 'no reasoning')

    # IoU override — R8 mitigation
    accepted = llm_raw
    override = False
    if yolo_iou and yolo_iou > IOU_THRESHOLD and not llm_raw:
        accepted  = True
        override  = True
        reasoning = f"YOLO-IoU override ({yolo_iou:.3f}>{IOU_THRESHOLD}): {reasoning}"

    status = "ACCEPTED" + (" [YOLO override]" if override else "") if accepted else "REJECTED"
    print(f"     LLM: {'ACCEPT' if llm_raw else 'REJECT'} → Final: {status}")
    print(f"     Reasoning: {reasoning[:90]}")

    return {**state,
            'explanation_accepted':    accepted,
            'llm_verdict_raw':         llm_raw,
            'yolo_override_triggered': override,
            'judge_reasoning':         reasoning,
            'iteration':               state['iteration'] + 1}

# Agent 3: Clinical Reporting Agent
def clinical_reporting_agent(state):
    """
    Qualitative demo only — not formally evaluated.
    Produces a structured natural language summary as described in the proposal.
    """
    print(f"  [Agent 3 — Clinical Reporting Agent]")
    iou_str = f"{state['iou_score']:.3f}" if state['iou_score'] else "N/A"

    prompt = f"""You are a clinical decision support AI assistant.
Brain MRI diagnosis:
  Classification: {state['predicted_class']}
  Confidence: {state['confidence']*100:.1f}%
  XAI method: {state['xai_method']} on {state['xai_layer']}
  Explanation spatial IoU vs ground truth: {iou_str}
  Explanation accepted: {state['explanation_accepted']}
  Clinical reasoning: {state['judge_reasoning']}

Write a brief structured clinical summary. Respond ONLY in JSON:
{{
  "abnormality_detected": true or false,
  "tumour_type": "tumour type or none",
  "confidence_level": "high/medium/low",
  "clinical_suggestion": "one sentence for the clinician",
  "recommended_action": "next clinical step"
}}

Note: This is a research demonstration only, not a clinical diagnosis."""

    raw    = query_local_llm(state['overlay_b64'], prompt)
    result = parse_llm_json(raw)
    print(f"     Abnormality: {result.get('abnormality_detected')}")
    print(f"     Suggestion:  {result.get('clinical_suggestion','')[:90]}")

    return {**state,
            'abnormality_detected': result.get('abnormality_detected'),
            'clinical_suggestion':  result.get('clinical_suggestion',
                                               state['predicted_class']),
            'final_confidence':     state['confidence']}

#  Routing
def should_retry(state):
    accepted  = state.get('explanation_accepted', False)
    iteration = state.get('iteration', 0)
    exhausted = len(state.get('tried_methods',[])) >= len(XAI_CONFIGS)
    if accepted:
        print("     → accepted — proceeding to Clinical Reporting")
        return "proceed"
    elif iteration >= MAX_ITER or exhausted:
        print("     → max iterations reached — proceeding anyway")
        return "proceed"
    else:
        print("     → rejected — retrying next XAI config")
        return "retry"

#  Compile graph
graph = StateGraph(AgentXAIState)
graph.add_node("xai_selector",       xai_selector_agent)
graph.add_node("llm_vision_judge",   llm_vision_judge_agent)
graph.add_node("clinical_reporting", clinical_reporting_agent)
graph.set_entry_point("xai_selector")
graph.add_edge("xai_selector", "llm_vision_judge")
graph.add_conditional_edges("llm_vision_judge", should_retry,
    {"retry":"xai_selector","proceed":"clinical_reporting"})
graph.add_edge("clinical_reporting", END)
app = graph.compile()

print("MRI-Xplain pipeline compiled")
print(f"  XAI search space: {len(XAI_CONFIGS)} configurations")
print(f"  IoU override threshold: {IOU_THRESHOLD}")
print(f"  Max iterations: {MAX_ITER}")

MRI-Xplain pipeline compiled
  XAI search space: 6 configurations
  IoU override threshold: 0.25
  Max iterations: 5


# Demo 3 (Final Version, added Human-in-the-loop to Demo 2)

In [7]:
!pip install gradio -q

import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import cv2
import torch
import io
import base64
from PIL import Image
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


#  Visual constants

ACCEPT_COLOR = '#1B5E20'   # dark green
REJECT_COLOR = '#B71C1C'   # dark red
NEUTRAL_COLOR = '#616161'  # grey
torch.cuda.empty_cache()

def get_cam_configs():
    return [
        {'name': 'GradCAM (L-1)',   'cam': GradCAM,         'layer': eff_model.features[-1]},
        {'name': 'GradCAM (L-3)',   'cam': GradCAM,         'layer': eff_model.features[-3]},
        {'name': 'GradCAM++ (L-3)', 'cam': GradCAMPlusPlus, 'layer': eff_model.features[-3]},
        {'name': 'GradCAM++ (L-5)', 'cam': GradCAMPlusPlus, 'layer': eff_model.features[-5]},
        {'name': 'LayerCAM (L-3)',  'cam': LayerCAM,        'layer': eff_model.features[-3]},
    ]


def make_single_image(img_r, title, subtitle="", border_color=None):
    """Makes a single larger image panel with optional coloured border."""
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img_r)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    if subtitle:
        ax.set_xlabel(subtitle, fontsize=11, labelpad=8)
    ax.axis('off')
    if border_color:
        for sp in ax.spines.values():
            sp.set_edgecolor(border_color)
            sp.set_linewidth(4)
            sp.set_visible(True)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100,
                bbox_inches='tight', facecolor='white')
    plt.close()
    buf.seek(0)
    return Image.open(buf)


def make_final_trace(img_r, iterations, yolo_box):
    """
    Builds the pipeline trace as a capped grid (max 4 columns) so panels
    stay a consistent, legible size regardless of how many XAI iterations
    were run.
    """
    panels = [{'type': 'input'}]
    for it in iterations:
        panels.append({'type': 'iter', 'it': it})
    panels.append({'type': 'yolo'})

    n_panels = len(panels)
    n_cols = min(4, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4.6 * n_cols, 4.9 * n_rows))
    axes = np.atleast_2d(axes)
    flat_axes = axes.flatten()

    for idx, panel in enumerate(panels):
        ax = flat_axes[idx]

        if panel['type'] == 'input':
            ax.imshow(img_r)
            ax.set_title('Input MRI', fontsize=12, fontweight='bold')

        elif panel['type'] == 'iter':
            it = panel['it']
            iter_no = iterations.index(it) + 1
            ax.imshow(it['overlay'])
            iou_s = f"IoU = {it['yolo_iou']:.3f}" if it['yolo_iou'] else "IoU = N/A"
            color = ACCEPT_COLOR if it['accepted'] else REJECT_COLOR
            status = 'Accepted' if it['accepted'] else 'Rejected'
            ax.set_title(f"Iteration {iter_no}: {it['name']}",
                         fontsize=10, fontweight='bold')
            ax.set_xlabel(f"{iou_s}\n{status}", fontsize=10, color=color,
                          fontweight='bold', labelpad=8)
            for sp in ax.spines.values():
                sp.set_edgecolor(color)
                sp.set_linewidth(3)
                sp.set_visible(True)

        elif panel['type'] == 'yolo':
            ax.imshow(img_r)
            if yolo_box:
                x1, y1, x2, y2, yc = yolo_box
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=3, edgecolor=REJECT_COLOR, facecolor='none'))
                ax.text(x1, max(0, y1 - 6), f'tumor {yc:.2f}',
                        fontsize=10, color='white', fontweight='bold',
                        bbox=dict(facecolor=REJECT_COLOR, edgecolor='none', pad=2))
                ax.set_title('YOLO Validator', fontsize=12, fontweight='bold')
                ax.set_xlabel(f'Confidence: {yc:.2f}', fontsize=10, labelpad=8)
            else:
                ax.set_title('YOLO Validator', fontsize=12, fontweight='bold')
                ax.set_xlabel('No detection', fontsize=10, color=NEUTRAL_COLOR,
                              labelpad=8)

        ax.axis('off')

    for idx in range(n_panels, len(flat_axes)):
        flat_axes[idx].axis('off')
        flat_axes[idx].set_visible(False)

    plt.suptitle('MRI-Xplain - Agent Pipeline Trace',
                  fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=110,
                bbox_inches='tight', facecolor='white')
    plt.close()
    buf.seek(0)
    return Image.open(buf)


#  Report text formatting (plain, clinical style | no emoji/box art)

def format_classification(pred_class, conf, all_probs, pred_idx):
    lines = [
        "CLASSIFICATION RESULT",
        "-" * 40,
        f"Predicted Class:      {pred_class}",
        f"Confidence:            {conf * 100:.1f}%",
        "",
        "Class Probabilities:",
    ]
    for i in range(4):
        marker = "  ->" if i == pred_idx else "    "
        lines.append(f"{marker} {CLASS_NAMES[i]:<14} {all_probs[i] * 100:5.1f}%")
    return "\n".join(lines) + "\n"


def format_xai_selector(iterations, best_name, best_iou):
    lines = [
        "AGENT 1 - XAI Selector",
        "-" * 40,
        f"Configurations evaluated: {len(iterations)} of 5",
        "",
    ]

    for i, it in enumerate(iterations):
        iou_s = f"{it['yolo_iou']:.3f}" if it['yolo_iou'] else "N/A"
        status = "Accepted" if it['accepted'] else "Rejected (below threshold)"
        lines.append(f"Iteration {i + 1}")
        lines.append(f"  Method:      {it['name']}")
        lines.append(f"  YOLO IoU:    {iou_s}")
        lines.append(f"  Result:      {status}")
        lines.append("")

    iou_display = f"{best_iou:.3f}" if best_iou else "N/A"
    trusted = "Above threshold" if (best_iou and best_iou > IOU_THRESHOLD) else "Below threshold"

    lines += [
        "-" * 40,
        f"Selected Method:      {best_name}",
        f"Best IoU Score:       {iou_display}",
        f"Spatial Agreement:    {trusted}",
    ]
    return "\n".join(lines) + "\n"


def format_judge(llm_acc, override, final_ok, verdict):
    lines = [
        "AGENT 2 - LLM Vision Judge",
        "-" * 40,
        f"LLM Assessment:        {'Accepted' if llm_acc else 'Rejected'}",
        f"YOLO Override Applied: {'Yes' if override else 'No'}",
        f"Final Verdict:         {'Accepted' if final_ok else 'Rejected'}",
        f"Clinical Coherence:    {verdict.get('clinical_coherence', 'Unknown')}",
        "",
        "Reasoning:",
        f"  {verdict.get('reasoning', 'No reasoning provided')}",
        "",
        "Recommended Action:",
        f"  {verdict.get('recommended_action', 'N/A')}",
    ]
    return "\n".join(lines) + "\n"


def format_report(pred_class, conf, final_ok, iou, verdict):
    iou_str = f"{iou:.3f}" if iou else "N/A"
    trust_str = "Trusted" if final_ok else "Not trusted"

    lines = [
        "AGENT 3 - Clinical Reporting Agent",
        "(Research demonstration -- NOT for clinical use)",
        "-" * 40,
        f"Diagnosis:              {pred_class}",
        f"Confidence:             {conf * 100:.1f}%",
        f"Spatial Agreement (IoU): {iou_str}",
        f"Explanation Status:     {trust_str}",
        "",
        "Clinical Reasoning:",
        f"  {verdict.get('reasoning', '')}",
        "",
        "Recommended Action:",
        f"  {verdict.get('recommended_action', 'Consult with specialist')}",
        "",
        "-" * 40,
        "This output is generated for research demonstration purposes",
        "only and must not be used for clinical decision-making.",
    ]
    return "\n".join(lines) + "\n"


def format_human_review(decision):
    """Format the human review decision as a standalone box."""
    lines = [
        "HUMAN REVIEW",
        "-" * 40,
        f"Reviewer Decision:     {decision}",
        "",
    ]
    if decision == "Approved":
        lines.append("Status: Explanation accepted for clinical use in this case.")
        lines.append("The XAI output aligns with radiologist judgment.")
    else:
        lines.append("Status: Explanation flagged for re-evaluation.")
        lines.append("The XAI output requires further review before clinical use.")

    return "\n".join(lines) + "\n"


#  Pipeline state shared between the two-phase run

_pipeline_state = {}


#  Phase 1: run everything up to and including Agent 2, then pause

def run_pipeline(image):
    """
    Phase 1 of the pipeline.
    Streams updates through classification -> Agent 1 -> Agent 2,
    then pauses and surfaces the approval panel.
    Agent 3 report is written only after the human decides.
    """
    global _pipeline_state
    _pipeline_state = {}

    if image is None:
        yield (None, "Upload a brain MRI scan to begin.", "", "", "", "",
               gr.update(visible=False), gr.update(visible=False))
        return

    img_pil = Image.fromarray(image).convert('RGB')
    img_r   = np.array(img_pil.resize((224, 224)))
    img_rgb = img_r.astype(float) / 255.0

    yield (None,
           "Processing scan.\n\nStep 1 of 4: Classifying brain MRI...",
           "", "", "", "",
           gr.update(visible=True),
           gr.update(visible=False))

    #  Classify
    tensor = val_transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(eff_model(tensor), dim=1)[0]
    pred_idx   = probs.argmax().item()
    conf       = probs[pred_idx].item()
    pred_class = CLASS_NAMES[pred_idx]
    all_probs  = probs.cpu().numpy()

    clf_text  = format_classification(pred_class, conf, all_probs, pred_idx)
    input_img = make_single_image(img_r,
                                   f'Input MRI — {pred_class} ({conf*100:.0f}%)')

    yield (input_img, clf_text,
           "Step 2 of 4: Agent 1 evaluating explainability configurations...",
           "", "", "",
           gr.update(visible=True),
           gr.update(visible=False))

    #  YOLO
    is_tumor  = pred_class != 'No Tumor'
    yolo_mask = None
    yolo_box  = None

    if is_tumor:
        cv2.imwrite('/content/temp_demo.jpg',
                     cv2.cvtColor(img_r, cv2.COLOR_RGB2BGR))
        res = yolo_detector('/content/temp_demo.jpg',
                            imgsz=224, conf=0.15, verbose=False)
        if res and len(res[0].boxes) > 0:
            boxes = res[0].boxes
            best  = boxes.conf.argmax().item()
            bx    = boxes.xyxy[best].cpu().numpy().astype(int)
            yc    = float(boxes.conf[best].item())
            yolo_mask = np.zeros((224, 224), dtype=np.float32)
            yolo_mask[max(0, bx[1]):min(224, bx[3]),
                      max(0, bx[0]):min(224, bx[2])] = 1.0
            yolo_box = (int(bx[0]), int(bx[1]),
                        int(bx[2]), int(bx[3]), yc)

    #  Agent 1: XAI iterations
    iterations = []
    best_it    = None
    best_iou   = -1

    for ci, cfg in enumerate(get_cam_configs()):
        try:
            with cfg['cam'](model=eff_model,
                            target_layers=[cfg['layer']]) as cam:
                h = cam(input_tensor=tensor,
                        targets=[ClassifierOutputTarget(pred_idx)])[0]
        except:
            continue

        h_n      = (h - h.min()) / (h.max() - h.min() + 1e-8)
        heat_rgb = plt.cm.jet(h_n)[:, :, :3]
        overlay  = np.clip(0.45 * img_rgb + 0.55 * heat_rgb, 0, 1)

        yolo_iou = None
        if yolo_mask is not None:
            hb    = (h_n > 0.2).astype(float)
            inter = (hb * yolo_mask).sum()
            union = np.clip(hb + yolo_mask, 0, 1).sum()
            yolo_iou = float(inter / (union + 1e-8))

        accepted = yolo_iou is not None and yolo_iou > IOU_THRESHOLD

        it = {'name': cfg['name'], 'overlay': overlay,
              'yolo_iou': yolo_iou, 'accepted': accepted}
        iterations.append(it)

        if yolo_iou is not None and yolo_iou > best_iou:
            best_iou = yolo_iou
            best_it  = it

        iter_img = make_single_image(
            (overlay * 255).astype(np.uint8),
            f'Iteration {ci+1}: {cfg["name"]}',
            f'IoU={f"{yolo_iou:.3f}" if yolo_iou else "N/A"} — '
            f'{"Accepted" if accepted else "trying next..."}',
            border_color=ACCEPT_COLOR if accepted else REJECT_COLOR
        )

        xai_progress = format_xai_selector(
            iterations,
            best_it['name'] if best_it else 'N/A',
            best_iou if best_iou > 0 else None
        )

        yield (iter_img, clf_text, xai_progress,
               f"Step 2 of 4: Agent 1, iteration {ci+1} of 5.\n"
               f"{'Acceptable explanation found.' if accepted else 'Evaluating next configuration...'}",
               "", "",
               gr.update(visible=True),
               gr.update(visible=False))

        if accepted:
            break

    if best_it is None and iterations:
        best_it = iterations[-1]

    iou = best_it['yolo_iou'] if best_it else None

    xai_text  = format_xai_selector(
        iterations,
        best_it['name'] if best_it else 'N/A',
        iou
    )
    trace_img = make_final_trace(img_r, iterations, yolo_box)

    yield (trace_img, clf_text, xai_text,
           "Step 3 of 4: Agent 2 (LLM Vision Judge) evaluating clinical "
           "coherence. This takes approximately 8 seconds...",
           "", "",
           gr.update(visible=True),
           gr.update(visible=False))

    #  Agent 2: LLM Judge
    buf = io.BytesIO()
    img_pil.resize((224, 224)).save(buf, format='JPEG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    prompt = (
        f"Brain MRI classified as: {pred_class} ({conf*100:.1f}%). "
        f"XAI: {best_it['name'] if best_it else 'N/A'}. "
        f"YOLO IoU: {f'{iou:.3f}' if iou else 'N/A'}. "
        f"Is the highlight clinically plausible? "
        f"Respond ONLY in JSON: "
        f"{{\"accepted\": true/false, "
        f"\"reasoning\": \"one sentence\", "
        f"\"clinical_coherence\": \"high/medium/low\", "
        f"\"recommended_action\": \"next step\"}}"
    )

    raw     = query_local_llm(b64, prompt)
    verdict = parse_llm_json(raw)
    llm_acc = bool(verdict.get('accepted', False))

    override = False
    final_ok = llm_acc
    if iou and iou > IOU_THRESHOLD and not llm_acc:
        override = True
        final_ok = True

    judge_text = format_judge(llm_acc, override, final_ok, verdict)
    report_text = format_report(pred_class, conf, final_ok, iou, verdict)

    # Save everything the human-review callbacks will need
    _pipeline_state = {
        'trace_img':  trace_img,
        'clf_text':   clf_text,
        'xai_text':   xai_text,
        'judge_text': judge_text,
        'report_text': report_text,
        'pred_class': pred_class,
        'conf':       conf,
        'final_ok':   final_ok,
        'iou':        iou,
        'verdict':    verdict,
    }

    # Yield with the processing banner hidden and the HITL panel visible
    # Agent 3 report is already shown; HITL buttons appear below
    yield (trace_img, clf_text, xai_text, judge_text, report_text, "",
           gr.update(visible=False),
           gr.update(visible=True))


#  Phase 2: human decision callbacks

def human_approve():
    """Called when the reviewer clicks Approve."""
    s = _pipeline_state
    if not s:
        return gr.update(), gr.update(visible=False)

    hitl_text = format_human_review("Approved")
    return hitl_text, gr.update(visible=False)


def human_reject():
    """Called when the reviewer clicks Reject."""
    s = _pipeline_state
    if not s:
        return gr.update(), gr.update(visible=False)

    hitl_text = format_human_review("Rejected")
    return hitl_text, gr.update(visible=False)


#  Gradio UI

custom_css = """
textarea {
    font-family: 'Courier New', 'Consolas', monospace !important;
    font-size: 12.5px !important;
    line-height: 1.45 !important;
    background-color: #fafafa !important;
}
.processing-banner {
    text-align: center;
    padding: 10px;
    background: #eef1f4;
    border: 1px solid #d5dbe0;
    border-radius: 6px;
    font-size: 13px;
    color: #37424a;
}
.hitl-panel {
    text-align: center;
    padding: 14px;
    background: #f9f4e8;
    border: 1px solid #d9c97a;
    border-radius: 6px;
    font-size: 13px;
    color: #4a3f00;
    margin-top: 6px;
}
"""

with gr.Blocks(title="MRI-Xplain", theme=gr.themes.Soft(),
               css=custom_css) as demo:

    gr.Markdown("""
    <div style="text-align:center; padding:18px;
                background:linear-gradient(135deg,#1a1a2e,#0f3460);
                border-radius:8px; margin-bottom:15px">
    <h1 style="color:white; margin:0; font-size:26px">MRI-Xplain</h1>
    <p style="color:#a0c4ff; margin:5px 0 0 0; font-size:14px">
    Agentic Framework for Faithful Brain Tumour Explanations</p>
    <p style="color:#7ecec4; margin:3px 0 0 0; font-size:12px">
    EfficientNetB0 &rarr; XAI Selector (Agent 1) &rarr;
    LLM Vision Judge (Agent 2) &rarr; Human Review &rarr; Clinical Report (Agent 3)</p>
    </div>
    """)

    processing_banner = gr.Markdown(
        "<div class='processing-banner'>Analysing scan — please wait.</div>",
        visible=False
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            img_in  = gr.Image(label="Upload Brain MRI Scan",
                               type="numpy", height=260)
            with gr.Row():
                run_btn    = gr.Button("Run Agent Pipeline",
                                       variant="primary", size="lg")
                cancel_btn = gr.Button("Cancel", variant="stop", size="lg")
            gr.Markdown("""
            **Pipeline Steps**
            1. EfficientNetB0 classifies the scan
            2. Agent 1 evaluates up to five XAI configurations
            3. YOLO independently validates tumour location
            4. Agent 2 (LLaVA) judges clinical coherence
            5. Agent 3 produces a summary report
            6. Human reviewer approves or rejects the explanation

            **Classes**
            Glioma, Meningioma, No Tumor, Pituitary

            Each output panel below is scrollable.
            """)

        with gr.Column(scale=3):
            gr.Markdown("### Pipeline Trace - XAI iterations and YOLO validation")
            trace_out = gr.Image(label="Agent Pipeline Trace",
                                  show_label=False, height=450)

    gr.Markdown("---")
    gr.Markdown(
        "### Agent Outputs  "
        "<span style='color:gray; font-size:12px'>(each box is scrollable)</span>"
    )

    with gr.Row(equal_height=True):
        clf_out = gr.Textbox(
            label="Classification",
            lines=12, max_lines=12,
        )
        xai_out = gr.Textbox(
            label="Agent 1 - XAI Selector",
            lines=12, max_lines=12,
        )

    with gr.Row(equal_height=True):
        judge_out = gr.Textbox(
            label="Agent 2 - LLM Vision Judge",
            lines=12, max_lines=12,
        )
        report_out = gr.Textbox(
            label="Agent 3 - Clinical Reporting Agent",
            lines=12, max_lines=12,
        )

    # Human Review Panel (hidden until Agent 3 report is ready)
    with gr.Group(visible=False) as hitl_panel:
        gr.Markdown("""
        <div class='hitl-panel'>
        <strong>Step 4: Human Reviewer Assessment</strong><br>
        Review the pipeline trace, Agent 2 judgment, and clinical report above.
        Then approve or reject the explanation before proceeding.
        </div>
        """)
        with gr.Row():
            approve_btn = gr.Button("Approve Explanation",
                                     variant="primary", size="lg")
            reject_btn  = gr.Button("Reject Explanation",
                                     variant="stop",    size="lg")

    # Human Review Decision Box (populated after approve/reject is clicked)
    hitl_out = gr.Textbox(
        label="Human Review Decision",
        lines=8, max_lines=8,
        visible=False
    )

    gr.Markdown("""
    <div style="text-align:center; color:gray; font-size:11px;
                margin-top:12px; padding:8px;
                border-top:1px solid #eee">
    Research demonstration only - outputs are not clinical diagnoses.<br>
    Imane Belbachir | MSc Computer Science with AI | Abertay University 2026
    </div>
    """)

    # Wire up the run / cancel buttons
    run_event = run_btn.click(
        fn=run_pipeline,
        inputs=[img_in],
        outputs=[trace_out, clf_out, xai_out, judge_out, report_out,
                 hitl_out, processing_banner, hitl_panel],
        show_progress="full"
    )
    cancel_btn.click(fn=None, cancels=[run_event])

    # Wire up the human review buttons
    approve_btn.click(
        fn=human_approve,
        inputs=[],
        outputs=[hitl_out, hitl_panel]
    ).then(
        fn=lambda: gr.update(visible=True),
        outputs=[hitl_out]
    )
    reject_btn.click(
        fn=human_reject,
        inputs=[],
        outputs=[hitl_out, hitl_panel]
    ).then(
        fn=lambda: gr.update(visible=True),
        outputs=[hitl_out]
    )

demo.launch(share=True, debug=False)


# ensure steps 1 to 5 running already.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://128866d8a081171aa9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# END